In [1]:
%pip install -q huggingface_hub

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

repo_id = "Qwen/Qwen3.5-4B"
model_dir = "/content/drive/MyDrive/NewsLens/models/Qwen3.5-4B"

revision = HfApi().model_info(repo_id).sha

snapshot_download(
    repo_id=repo_id,
    revision=revision,
    local_dir=model_dir,
    max_workers=2,
)

Path(model_dir, "download_revision.txt").write_text(revision)

print("Download complete:", model_dir)
print("Model revision:", revision)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Download complete: /content/drive/MyDrive/NewsLens/models/Qwen3.5-4B
Model revision: 851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a


# NewsLens: controlled Qwen3.5-4B experiment

The three original preparation cells above are preserved. Do not rerun the download cell when the fixed snapshot already exists. Use the T4 runtime. Execute each stage in order and inspect failures before continuing. Model weights and datasets stay outside Git.

## Upload the source bundle

Build `artifacts/newslens-source.zip` locally with `python scripts/build_bundle.py`, then select it below. Resume using the same source bundle and configuration.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
bundle_name = next(name for name in uploaded if name.endswith(".zip"))
workspace = Path("/content/newslens")
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle_name) as archive:
    for name in archive.namelist():
        if not (workspace / name).resolve().is_relative_to(workspace.resolve()):
            raise ValueError("Unsafe source bundle entry")
    archive.extractall(workspace)
os.chdir(workspace)
print("Source ready:", workspace)


## Install the compatible GPU stack

Run once in a fresh runtime, then choose **Runtime → Restart session** before continuing. On later sessions, reinstall the package versions and Git revisions captured by the passing smoke test. The installer starts from the official Unsloth T4 recipe.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/install_colab.py"], check=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, subprocess, sys
os.chdir("/content/newslens")
ROOT = Path("/content/drive/MyDrive/NewsLens")
DATA = ROOT / "data" / "prepared-v1"
RUN = ROOT / "runs" / "qwen35-4b-v1"
MODEL = Path("/content/models/Qwen3.5-4B")
def run(stage, *extra):
    subprocess.run([sys.executable, "-u", "-m", "newslens", stage,
        "--data-dir", str(DATA), "--run-dir", str(RUN), "--model-dir", str(MODEL), *extra], check=True)
run("stage-model")
run("prepare")


## Disposable compatibility gate

This performs 50 optimizer steps with a deliberate checkpoint stop/resume at step 25. It verifies frozen weights, adapter updates, finite gradients, reload equivalence, and GPU headroom. Its adapter is discarded from the actual experiment. Read the measured duration estimate before the full run; free Colab sessions may require multiple resumptions.


In [ ]:
run("smoke-test")
import json
gate = json.loads((RUN / "smoke" / "gate.json").read_text())
assert gate["passed"]
print({k: gate[k] for k in ("estimated_training_hours_excluding_validation", "estimated_evaluation_hours", "memory")})


## Unchanged-model baseline and prompt experiments

Runs all six frozen prompt conditions against the full official test set. Predictions persist every 100 examples. Repeating this stage resumes matching incomplete files.


In [ ]:
run("evaluate")


## Full LoRA training

Starts from the original snapshot. All six unchanged-model evaluations must be complete. Recovery archives include optimizer/scheduler/RNG state. Repeating this stage restores the latest complete checkpoint. Full validation loss selects the best epoch checkpoint.


In [ ]:
run("train")


## LoRA test evaluation and measured report

In [ ]:
run("evaluate", "--adapter", str(RUN / "adapter"))
run("report", "--output-dir", str(RUN / "results"))
from IPython.display import Markdown, display
display(Markdown((RUN / "results" / "REPORT.md").read_text()))


In [ ]:
examples = [
    "Diplomats from several nations agreed to resume peace negotiations next week.",
    "The national team won the final after a dramatic penalty shootout.",
    "The retailer reported higher quarterly profits as online sales grew.",
    "Researchers developed a new chip design that reduces computing energy use.",
]
for text in examples:
    run("predict", "--adapter", str(RUN / "adapter"), "--text", text)


## Export artifacts

Keep the base snapshot and recovery archives in Drive. Export the adapter, measured predictions, run metadata, GPU lock file, and generated report for final verification and GitHub publication. Download this executed notebook using File → Download → Download .ipynb.


In [ ]:
import zipfile
from google.colab import files
export = Path("/content/newslens-results.zip")
with zipfile.ZipFile(export, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for relative in ("adapter", "results", "predictions"):
        for path in (RUN / relative).rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(RUN))
    for relative in ("protocol.json", "execution_manifest.json", "smoke/gate.json", "smoke/requirements-gpu.lock.txt", "training/summary.json"):
        archive.write(RUN / relative, relative)
files.download(str(export))
